# HOL 2: Data Exploration with Cortex Code

## Using AI to Generate Insights from Your Data

In this lab, you'll use **Cortex Code** (the AI assistant in Snowsight) to generate exploratory data analysis code. Instead of writing code from scratch, you'll provide natural language prompts and let the AI generate the analysis for you.

**What you'll learn:**
- How to use Cortex Code to generate Python/SQL analysis code
- Effective prompting techniques for data exploration
- Quick ways to gain insights from business data

**How to use Cortex Code in this notebook:**
1. Click on a new cell
2. Click the **Cortex Code icon** (sparkle icon) or press the AI shortcut
3. Type your prompt in natural language
4. Review and run the generated code

**Prerequisites:** You must have completed HOL 1 (Gold layer tables must exist).

---

In [ ]:
# Setup: Run this cell first to establish the session
from snowflake.snowpark.context import get_active_session
import snowflake.snowpark.functions as F

session = get_active_session()
session.use_database('BB_TRAINING')
session.use_warehouse('BB_TRAINING_WH')

print("Session ready! You can now use Cortex Code to explore the data.")
print("\nAvailable Gold tables:")
print("  - GOLD.GOLD_LOAN_PERFORMANCE  (per-loan metrics)")
print("  - GOLD.GOLD_CUSTOMER_360      (customer-level view)")
print("  - GOLD.GOLD_PORTFOLIO_SUMMARY  (monthly aggregates)")

---
## Prompt 1: Portfolio Distribution by Industry and Risk

**Instructions:** Create a new code cell below and use Cortex Code with this prompt:

> Using the BB_TRAINING.GOLD.GOLD_CUSTOMER_360 table, create a visualization showing the distribution of total loan exposure (TOTAL_EXPOSURE) broken down by INDUSTRY and CUSTOMER_RISK_CATEGORY. Use a grouped bar chart with matplotlib or plotly. Show the total exposure in millions on the y-axis and industries on the x-axis, with different colors for each risk category. Add a title and legend.

**What to look for:** Which industries have the highest concentration of high-risk customers?

---

In [ ]:
# PROMPT 1: Use Cortex Code here (click the AI icon and paste the prompt above)
# Or if Cortex Code is not available, run this fallback:

import matplotlib.pyplot as plt
import pandas as pd

df = session.sql("""
    SELECT INDUSTRY, CUSTOMER_RISK_CATEGORY, 
           SUM(TOTAL_EXPOSURE) / 1000000 as EXPOSURE_MILLIONS
    FROM BB_TRAINING.GOLD.GOLD_CUSTOMER_360
    WHERE CUSTOMER_RISK_CATEGORY != 'No Loan History'
    GROUP BY INDUSTRY, CUSTOMER_RISK_CATEGORY
    ORDER BY INDUSTRY
""").to_pandas()

pivot_df = df.pivot(index='INDUSTRY', columns='CUSTOMER_RISK_CATEGORY', values='EXPOSURE_MILLIONS').fillna(0)
pivot_df.plot(kind='bar', figsize=(12, 6), width=0.8)
plt.title('Loan Exposure by Industry and Risk Category', fontsize=14)
plt.ylabel('Total Exposure ($M)')
plt.xlabel('Industry')
plt.legend(title='Risk Category')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

---
## Prompt 2: Monthly Application Trends and Approval Rates

**Instructions:** Create a new code cell below and use Cortex Code with this prompt:

> Query BB_TRAINING.GOLD.GOLD_PORTFOLIO_SUMMARY and create a dual-axis line chart showing monthly TOTAL_APPLICATIONS (left y-axis, as a bar chart) and APPROVAL_RATE (right y-axis, as a line) over time (APPLICATION_MONTH). Aggregate across all industries. Highlight any months where the approval rate dropped below 60%. Use matplotlib with a clean style.

**What to look for:** Are there seasonal patterns in application volume? Do approval rates change over time?

---

In [ ]:
# PROMPT 2: Use Cortex Code here, or run the fallback below

import matplotlib.pyplot as plt
import pandas as pd

df = session.sql("""
    SELECT APPLICATION_MONTH,
           SUM(TOTAL_APPLICATIONS) as TOTAL_APPLICATIONS,
           SUM(TOTAL_APPROVALS) as TOTAL_APPROVALS,
           ROUND(SUM(TOTAL_APPROVALS) / NULLIF(SUM(TOTAL_APPLICATIONS), 0) * 100, 1) as APPROVAL_RATE
    FROM BB_TRAINING.GOLD.GOLD_PORTFOLIO_SUMMARY
    GROUP BY APPLICATION_MONTH
    ORDER BY APPLICATION_MONTH
""").to_pandas()

df['APPLICATION_MONTH'] = pd.to_datetime(df['APPLICATION_MONTH'])

fig, ax1 = plt.subplots(figsize=(14, 6))

# Bar chart for applications
ax1.bar(df['APPLICATION_MONTH'], df['TOTAL_APPLICATIONS'], width=20, alpha=0.6, color='steelblue', label='Applications')
ax1.set_xlabel('Month')
ax1.set_ylabel('Total Applications', color='steelblue')
ax1.tick_params(axis='y', labelcolor='steelblue')

# Line chart for approval rate
ax2 = ax1.twinx()
ax2.plot(df['APPLICATION_MONTH'], df['APPROVAL_RATE'], color='darkred', linewidth=2, marker='o', markersize=4, label='Approval Rate')
ax2.set_ylabel('Approval Rate (%)', color='darkred')
ax2.tick_params(axis='y', labelcolor='darkred')
ax2.axhline(y=60, color='red', linestyle='--', alpha=0.5, label='60% threshold')

plt.title('Monthly Application Volume and Approval Rate', fontsize=14)
fig.legend(loc='upper left', bbox_to_anchor=(0.1, 0.95))
plt.tight_layout()
plt.show()

---
## Prompt 3: Delinquency Patterns by Segment

**Instructions:** Create a new code cell below and use Cortex Code with this prompt:

> Using BB_TRAINING.GOLD.GOLD_LOAN_PERFORMANCE, calculate the delinquency rate (percentage of loans with LATE_PAYMENT_COUNT > 0) grouped by RISK_TIER and LOAN_PRODUCT. Display as a heatmap using matplotlib/seaborn showing the delinquency rate percentage in each cell. Use a red color gradient where darker = higher delinquency.

**What to look for:** Which combination of risk tier and loan product has the highest delinquency?

---

In [ ]:
# PROMPT 3: Use Cortex Code here, or run the fallback below

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

df = session.sql("""
    SELECT RISK_TIER, LOAN_PRODUCT,
           COUNT(*) as TOTAL_LOANS,
           SUM(CASE WHEN LATE_PAYMENT_COUNT > 0 THEN 1 ELSE 0 END) as DELINQUENT_LOANS,
           ROUND(SUM(CASE WHEN LATE_PAYMENT_COUNT > 0 THEN 1 ELSE 0 END) / COUNT(*) * 100, 1) as DELINQUENCY_RATE
    FROM BB_TRAINING.GOLD.GOLD_LOAN_PERFORMANCE
    WHERE RISK_TIER IS NOT NULL
    GROUP BY RISK_TIER, LOAN_PRODUCT
    ORDER BY RISK_TIER, LOAN_PRODUCT
""").to_pandas()

# Create pivot for heatmap
pivot = df.pivot(index='RISK_TIER', columns='LOAN_PRODUCT', values='DELINQUENCY_RATE').fillna(0)

# Reorder risk tiers
tier_order = ['Low', 'Medium', 'High', 'Very High']
pivot = pivot.reindex([t for t in tier_order if t in pivot.index])

fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(pivot.values, cmap='Reds', aspect='auto')

# Labels
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns, rotation=45, ha='right')
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)

# Add text annotations
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        text = ax.text(j, i, f"{pivot.values[i, j]:.1f}%",
                      ha="center", va="center", color="black" if pivot.values[i, j] < 50 else "white")

plt.colorbar(im, label='Delinquency Rate (%)')
plt.title('Delinquency Rate by Risk Tier and Loan Product', fontsize=14)
plt.xlabel('Loan Product')
plt.ylabel('Risk Tier')
plt.tight_layout()
plt.show()

---
## Prompt 4: Loan Size vs Default Rate by Product

**Instructions:** Create a new code cell below and use Cortex Code with this prompt:

> From BB_TRAINING.GOLD.GOLD_LOAN_PERFORMANCE, create a scatter plot comparing average LOAN_AMOUNT (x-axis) vs delinquency rate (y-axis, calculated as % of loans with LATE_PAYMENT_COUNT > 0), grouped by LOAN_PRODUCT. Each point should be a loan product, sized by the number of loans. Add labels for each point. Use matplotlib. Is there a relationship between loan size and default rate?

**What to look for:** Do larger loans have higher or lower default rates? Which products are outliers?

---

In [ ]:
# PROMPT 4: Use Cortex Code here, or run the fallback below

import matplotlib.pyplot as plt
import pandas as pd

df = session.sql("""
    SELECT LOAN_PRODUCT,
           AVG(LOAN_AMOUNT) as AVG_LOAN_AMOUNT,
           COUNT(*) as LOAN_COUNT,
           ROUND(SUM(CASE WHEN LATE_PAYMENT_COUNT > 0 THEN 1 ELSE 0 END) / COUNT(*) * 100, 1) as DELINQUENCY_RATE
    FROM BB_TRAINING.GOLD.GOLD_LOAN_PERFORMANCE
    GROUP BY LOAN_PRODUCT
""").to_pandas()

fig, ax = plt.subplots(figsize=(10, 7))

# Scatter with size proportional to loan count
scatter = ax.scatter(
    df['AVG_LOAN_AMOUNT'] / 1000,  # Convert to thousands
    df['DELINQUENCY_RATE'],
    s=df['LOAN_COUNT'] * 2,  # Size by count
    alpha=0.7,
    c=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd'],
    edgecolors='black'
)

# Add labels
for _, row in df.iterrows():
    ax.annotate(row['LOAN_PRODUCT'], 
                (row['AVG_LOAN_AMOUNT']/1000, row['DELINQUENCY_RATE']),
                textcoords="offset points", xytext=(10, 5), fontsize=10)

ax.set_xlabel('Average Loan Amount ($K)', fontsize=12)
ax.set_ylabel('Delinquency Rate (%)', fontsize=12)
ax.set_title('Loan Size vs Delinquency Rate by Product', fontsize=14)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nSummary:")
print(df[['LOAN_PRODUCT', 'AVG_LOAN_AMOUNT', 'LOAN_COUNT', 'DELINQUENCY_RATE']].to_string(index=False))

---
## Prompt 5: Top Customers by Exposure with Health Scores

**Instructions:** Create a new code cell below and use Cortex Code with this prompt:

> Query BB_TRAINING.GOLD.GOLD_CUSTOMER_360 to show the top 15 customers by TOTAL_EXPOSURE. Create a horizontal bar chart with BUSINESS_NAME on the y-axis and TOTAL_EXPOSURE on the x-axis. Color-code the bars by CUSTOMER_RISK_CATEGORY (green for Low Risk, orange for Medium Risk, red for High Risk). Add the AVG_PAYMENT_HEALTH score as text annotation at the end of each bar. Format exposure values in millions.

**What to look for:** Are our highest-exposure customers also our highest-risk? Where should we focus attention?

---

In [ ]:
# PROMPT 5: Use Cortex Code here, or run the fallback below

import matplotlib.pyplot as plt
import pandas as pd

df = session.sql("""
    SELECT BUSINESS_NAME, INDUSTRY, TOTAL_EXPOSURE, 
           AVG_PAYMENT_HEALTH, CUSTOMER_RISK_CATEGORY
    FROM BB_TRAINING.GOLD.GOLD_CUSTOMER_360
    WHERE TOTAL_EXPOSURE > 0
    ORDER BY TOTAL_EXPOSURE DESC
    LIMIT 15
""").to_pandas()

# Color mapping
colors = {
    'Low Risk': '#2ecc71',
    'Medium Risk': '#f39c12', 
    'High Risk': '#e74c3c'
}
bar_colors = [colors.get(r, '#95a5a6') for r in df['CUSTOMER_RISK_CATEGORY']]

fig, ax = plt.subplots(figsize=(12, 8))

# Horizontal bar chart
bars = ax.barh(range(len(df)), df['TOTAL_EXPOSURE'] / 1_000_000, color=bar_colors, edgecolor='black', alpha=0.8)

# Labels
ax.set_yticks(range(len(df)))
ax.set_yticklabels(df['BUSINESS_NAME'], fontsize=9)
ax.set_xlabel('Total Exposure ($M)', fontsize=12)
ax.set_title('Top 15 Customers by Loan Exposure (Color = Risk Category)', fontsize=14)

# Add health score annotations
for i, (val, health) in enumerate(zip(df['TOTAL_EXPOSURE'], df['AVG_PAYMENT_HEALTH'])):
    if pd.notna(health):
        ax.text(val/1_000_000 + 0.05, i, f" Health: {health:.0f}", va='center', fontsize=8)

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=l) for l, c in colors.items()]
ax.legend(handles=legend_elements, loc='lower right')

ax.invert_yaxis()
plt.tight_layout()
plt.show()

---
## Summary

You've completed the data exploration section! Here's what you discovered:

1. **Industry Risk Concentration** - Which industries carry the most high-risk exposure
2. **Seasonal Patterns** - How application volumes and approval rates change over time
3. **Delinquency Drivers** - Which risk tier + product combinations are most problematic
4. **Size vs Risk** - Whether larger loans correlate with higher default rates
5. **Concentration Risk** - Whether our biggest customers are also our riskiest

### Key Takeaway
Using Cortex Code, you can generate analysis code in seconds that would otherwise take 15-30 minutes to write from scratch. The key is writing clear, specific prompts that describe the data source, the desired output, and the visual format.

### Next: Snowflake Intelligence
Continue to the SI section to build a natural language interface over this data!